# Session 3.4: Permutation Invariant Training (PIT) Loss

## The permutation problem

Your Conv-TasNet outputs `(batch, C, T)` — say 2 separated waveforms. Your dataset provides clean source waveforms `(batch, C, T)` as ground truth. Now you want to compute a loss.

The problem: **there's no guarantee the network's output order matches the ground truth order.**

```
Network output:    [waveform_A, waveform_B]
Ground truth:      [speaker_1,  speaker_2]
```

Maybe `waveform_A` sounds like speaker 1 and `waveform_B` sounds like speaker 2. Or maybe it's the reverse. The network has no reason to be consistent — nothing in the architecture forces output 0 to always be speaker 1. In fact, if you naively compute:

```
loss = SI-SDR(output[0], target[0]) + SI-SDR(output[1], target[1])
```

You might be penalizing the network for getting the *right* separation in the *wrong* order. It would learn nothing useful — gradients would be contradictory across batches.

## SI-SDR — the metric you optimize

Before PIT, you need to understand what you're measuring. Scale-Invariant Signal-to-Distortion Ratio measures how well a predicted waveform matches a target, regardless of volume scaling.

Given predicted signal `s_hat` and target signal `s`:

```
Step 1 — project prediction onto target:
    s_target = (<s_hat, s> / <s, s>) × s
    
    This is the component of s_hat that points in the same
    direction as s. The inner product <a,b> = sum(a*b).

Step 2 — compute the error (what's left over):
    e_noise = s_hat - s_target

Step 3 — SI-SDR:
    SI-SDR = 10 × log10(||s_target||² / ||e_noise||²)
```

Higher is better. 0 dB means signal and noise have equal power. 20 dB means signal is 100× stronger than error. The "scale-invariant" part means if your prediction is twice as loud as the target but otherwise perfect, you still get infinite SI-SDR — volume differences don't count as errors.

For training you want to **maximize** SI-SDR, which means **minimizing** `-SI-SDR` as a loss.

Concrete example with numbers:

```
target:    s  = [1.0,  2.0, -1.0,  0.5]
predicted: s_hat = [2.1,  4.2, -2.05, 1.1]   ← roughly 2× scaled version

<s_hat, s> = 2.1×1 + 4.2×2 + (-2.05)×(-1) + 1.1×0.5
           = 2.1 + 8.4 + 2.05 + 0.55 = 13.1

<s, s>     = 1² + 2² + 1² + 0.5² = 1 + 4 + 1 + 0.25 = 6.25

s_target   = (13.1 / 6.25) × [1.0, 2.0, -1.0, 0.5]
           = 2.096 × [1.0, 2.0, -1.0, 0.5]
           = [2.096, 4.192, -2.096, 1.048]

e_noise    = s_hat - s_target
           = [2.1-2.096, 4.2-4.192, -2.05-(-2.096), 1.1-1.048]
           = [0.004, 0.008, 0.046, 0.052]

||s_target||² = 2.096² + 4.192² + 2.096² + 1.048² ≈ 27.34
||e_noise||²  = 0.004² + 0.008² + 0.046² + 0.052² ≈ 0.0049

SI-SDR = 10 × log10(27.34 / 0.0049) ≈ 10 × log10(5579) ≈ 37.5 dB
```

High SI-SDR because the prediction is almost a perfect scaled copy of the target.

## The PIT algorithm

For `C` speakers, you try all `C!` permutations of the output-to-target assignment and pick the one that gives the lowest loss:

```
For C=2:
  Permutation 0: output[0]↔target[0], output[1]↔target[1]
  Permutation 1: output[0]↔target[1], output[1]↔target[0]

  loss_0 = -SI-SDR(output[0], target[0]) - SI-SDR(output[1], target[1])
  loss_1 = -SI-SDR(output[0], target[1]) - SI-SDR(output[1], target[0])

  final_loss = min(loss_0, loss_1)
```

For C=3 you'd try 6 permutations. For C=2 and C=3 this is cheap — no need for the Hungarian algorithm at this scale, brute force is fine.

This is **utterance-level PIT (uPIT)** — one permutation assignment per utterance, not per frame.

---

# TODO 5: Implement SI-SDR and uPIT Loss

**Goal:** Two functions — `si_sdr` and `pit_loss` — that together form the training loss for Conv-TasNet.

**Part A: `si_sdr(predicted, target)`**

```
Inputs:
  predicted: (batch, T)   — one speaker's predicted waveform
  target:    (batch, T)   — one speaker's ground truth waveform

Output:
  (batch,)   — SI-SDR value per sample in the batch
```

Algorithm:
```
1. Zero-mean both signals:
   target    = target - mean(target)
   predicted = predicted - mean(predicted)

2. Compute scaling factor:
   dot   = sum(predicted * target, dim=-1)
   power = sum(target * target, dim=-1) + 1e-8   ← small epsilon for stability

3. Project:
   s_target = (dot / power).unsqueeze(-1) * target

4. Compute noise:
   e_noise = predicted - s_target

5. SI-SDR:
   return 10 * log10(sum(s_target²) / (sum(e_noise²) + 1e-8))
```

**Part B: `pit_loss(outputs, targets)`**

```
Inputs:
  outputs: (batch, C, T)   — network's separated waveforms
  targets: (batch, C, T)   — ground truth clean sources

Output:
  scalar   — mean loss across batch (to call .backward() on)
```

Algorithm:
```
1. Generate all permutations of range(C)
   For C=2: [[0,1], [1,0]]
   For C=3: [[0,1,2], [0,2,1], [1,0,2], [1,2,0], [2,0,1], [2,1,0]]

2. For each permutation:
   compute mean SI-SDR across all C speakers for that assignment
   loss = -mean_si_sdr   ← negative because we minimize loss, maximize SI-SDR

3. Stack losses for all permutations: (num_permutations, batch)

4. Take minimum loss per sample: min across permutation dim → (batch,)

5. Return mean across batch
```

**Hints:**
- `itertools.permutations(range(C))` generates all permutations
- Keep operations batched — avoid Python loops over batch dimension
- The `1e-8` epsilon prevents log(0) and division by zero
- Zero-meaning the signals before computing SI-SDR is important — it removes DC offset which would otherwise dominate the loss

**Expected behavior:**
```python
# Perfect prediction → very high SI-SDR → very negative loss
target = torch.randn(4, 2, 480000)
perfect = target.clone()
loss = pit_loss(perfect, target)
print(loss)   # should be a very large negative number

# Silence prediction → terrible SI-SDR → large positive loss  
silence = torch.zeros(4, 2, 480000)
loss2 = pit_loss(silence, target)
print(loss2)  # should be much larger (worse)

# Verify permutation invariance — swapped outputs same loss as correct order
outputs = target.clone()
outputs_swapped = torch.stack([target[:,1,:], target[:,0,:]], dim=1)
loss3 = pit_loss(outputs, target)
loss4 = pit_loss(outputs_swapped, target)
print(torch.allclose(loss3, loss4))   # should be True
```

In [79]:
import torch
from itertools import permutations

# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

✅ Using Apple Silicon GPU
PyTorch version: 2.1.0
Device: mps


In [80]:
def si_sdr(predicted, target):
    epsilon = 1e-8
    
    predicted = predicted - torch.mean(predicted, dim=-1, keepdim=True)
    target = target - torch.mean(target, dim=-1, keepdim=True)
    
    dot = torch.sum(predicted * target, dim=-1)
    power = torch.sum(target * target, dim=-1) + epsilon
    
    s_target = (dot/power).unsqueeze(-1) * target
    e_noise = predicted - s_target
    
    return 10 * torch.log10(torch.sum(torch.pow(s_target, 2), dim=-1)/(torch.sum(torch.pow(e_noise, 2), dim=-1) + epsilon))

In [81]:
def pit_loss(outputs, targets):
    num_speaker = outputs.shape[1]
    perm_iterator = permutations(range(num_speaker))
    perm_list = list(perm_iterator)
    
    loss_list = []
    
    for perm in perm_list:
        speaker_losses = []
        for count, j in enumerate(perm):
            speaker_loss = si_sdr(outputs[:, j, :], targets[:, count, :])
            speaker_losses.append(speaker_loss)
        perm_loss = -torch.stack(speaker_losses, dim=0).mean(dim=0)
        loss_list.append(perm_loss)
    
    loss_stack = torch.stack(loss_list, dim=0)
    min_loss = torch.min(loss_stack, dim=0).values
    
    return min_loss.mean()


In [82]:
# Perfect prediction → very high SI-SDR → very negative loss
target = torch.randn(4, 2, 480000)
perfect = target.clone()
loss = pit_loss(perfect, target)
print(loss)   # should be a very large negative number

# Silence prediction → terrible SI-SDR → large positive loss  
silence = torch.zeros(4, 2, 480000)
loss2 = pit_loss(silence, target)
print(loss2)  # should be much larger (worse)

# Verify permutation invariance — swapped outputs same loss as correct order
outputs = target.clone()
outputs_swapped = torch.stack([target[:,1,:], target[:,0,:]], dim=1)
loss3 = pit_loss(outputs, target)
loss4 = pit_loss(outputs_swapped, target)
print(torch.allclose(loss3, loss4))   # should be True

tensor(-136.8066)
tensor(inf)
True
